# Data Reading

In [0]:
df=spark.read.format("parquet").load("abfss://bronze@databricksetlpractice.dfs.core.windows.net/products")

In [0]:
df.drop("_rescued_data").display()

product_id,product_name,category,brand,price
P0001,Clearly Its,Beauty,Nike,1868.54
P0002,Production Clear,Beauty,Apple,587.13
P0003,Culture Coach,Home,Revlon,1599.24
P0004,Movement Part,Sports,LG,651.71
P0005,Fact Name,Clothing,Samsung,1861.78
P0006,Usually Stop,Toys,Adidas,936.36
P0007,Reveal Current,Sports,Adidas,1954.02
P0008,Force Language,Beauty,Puma,1251.26
P0009,Stage Leg,Clothing,Samsung,1247.15
P0010,Leader Then,Sports,Sony,975.53


# Functions

In [0]:
df.createOrReplaceTempView("products")

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_catalog.bronze.discountt_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.95;



In [0]:
%sql

SELECT product_id,
       databricks_catalog.bronze.discountt_func(price) AS discounted_price
FROM products;

product_id,discounted_price
P0001,1775.1129999999998
P0002,557.7735
P0003,1519.278
P0004,619.1245
P0005,1768.6909999999998
P0006,889.5419999999999
P0007,1856.319
P0008,1188.697
P0009,1184.7925
P0010,926.7534999999999


In [0]:
from pyspark.sql.functions import expr
df = df.withColumn(

    "discounted_price",
    expr("databricks_catalog.bronze.discountt_func(price)")
)

df.display()

product_id,product_name,category,brand,price,_rescued_data,discounted_price
P0001,Clearly Its,Beauty,Nike,1868.54,null,1775.1129999999998
P0002,Production Clear,Beauty,Apple,587.13,null,557.7735
P0003,Culture Coach,Home,Revlon,1599.24,null,1519.278
P0004,Movement Part,Sports,LG,651.71,null,619.1245
P0005,Fact Name,Clothing,Samsung,1861.78,null,1768.6909999999998
P0006,Usually Stop,Toys,Adidas,936.36,null,889.5419999999999
P0007,Reveal Current,Sports,Adidas,1954.02,null,1856.319
P0008,Force Language,Beauty,Puma,1251.26,null,1188.697
P0009,Stage Leg,Clothing,Samsung,1247.15,null,1184.7925
P0010,Leader Then,Sports,Sony,975.53,null,926.7534999999999


In [0]:
%sql

CREATE OR REPLACE FUNCTION databricks_catalog.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
    return p_brand.upper()
$$;
from products

product_id,product_name,category,brand,price,_rescued_data
P0001,Clearly Its,Beauty,Nike,1868.54,null
P0002,Production Clear,Beauty,Apple,587.13,null
P0003,Culture Coach,Home,Revlon,1599.24,null
P0004,Movement Part,Sports,LG,651.71,null
P0005,Fact Name,Clothing,Samsung,1861.78,null
P0006,Usually Stop,Toys,Adidas,936.36,null
P0007,Reveal Current,Sports,Adidas,1954.02,null
P0008,Force Language,Beauty,Puma,1251.26,null
P0009,Stage Leg,Clothing,Samsung,1247.15,null
P0010,Leader Then,Sports,Sony,975.53,null


In [0]:
%sql

SELECT brand,
       databricks_catalog.bronze.upper_func(brand) AS upper_brand
FROM products;

brand,upper_brand
Nike,NIKE
Apple,APPLE
Revlon,REVLON
LG,LG
Samsung,SAMSUNG
Adidas,ADIDAS
Adidas,ADIDAS
Puma,PUMA
Samsung,SAMSUNG
Sony,SONY


In [0]:
df.write.format("delta") \
    .mode("append") \
    .option("path", "abfss://silver@databricksetlpractice.dfs.core.windows.net/products/") \
    .save()